# HuggingFace: Running Local LLMs Without Ollama

Ollama is great for daily use, but the **HuggingFace Transformers** library is the **industry standard** for research and production ML. Think of it this way:

- **Ollama** = Your phone camera (easy, convenient, great results)
- **HuggingFace** = A professional DSLR camera (more control, more complexity, used by pros)

### Why learn HuggingFace?

1. **Fine-tuning**: You can't fine-tune models through Ollama. HuggingFace is required.
2. **Model architecture access**: You can inspect, modify, and experiment with the actual neural network layers.
3. **Ecosystem**: 500,000+ models on HuggingFace Hub — the largest ML model repository in the world.
4. **Research**: Every new model paper releases code using HuggingFace.
5. **Deployability**: Production systems (AWS SageMaker, Azure ML, etc.) all support HuggingFace models natively.

In this notebook, we'll run a small model (GPT-2) to keep things fast and demonstrate the core concepts.

In [1]:
!pip install transformers accelerate torch -q

## 1. The Pipeline API: Easiest Entry Point

The `pipeline()` function is HuggingFace's "just make it work" API. It automatically:
1. Downloads the model from HuggingFace Hub
2. Loads the right tokenizer
3. Sets up the inference pipeline
4. Handles all the tensor math behind the scenes

First download will take a minute (it caches the model locally for future use).

In [2]:
from transformers import pipeline

# Using GPT-2 (124M params) — small enough to run on any hardware
generator = pipeline('text-generation', model='gpt2')

response = generator(
    "The history of pizza begins in", 
    max_length=60, 
    num_return_sequences=1
)
print(response[0]['generated_text'])

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'max_length', 'num_return_sequences'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=60) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The history of pizza begins in the mid-20th century. In the 18th century, the invention of the pizza oven and the rise of the pizza oven led to a rise in the popularity of the pizza. However, what started as a niche industry continued to grow and the popularity of pizza has continued to increase. In the 1920s, pizza was the standard in many areas of life in America. That year, the American Pizza Association developed the first pizzeria franchise and it continues to provide pizzerias nationwide. The first pizza was unveiled in the United States in 1930.

In the 20th century, the popularity of pizza began to increase as the popularity of pizza increased. It became an everyday activity of many people in the United States. The first pizza was introduced in Philadelphia. The first pizza shop opened in the Philadelphia area in 1885 and it continues to serve pizza for many years to come.

Today, the popularity of pizza continues to grow. The popularity of pizza continues to grow and the popul

**Important context:** GPT-2 is from 2019 and has only 124M parameters. It's not instruction-tuned, so it won't follow instructions like modern models. It simply *completes* text — give it a beginning and it continues. This is what ALL LLMs do at their core, before instruction tuning makes them conversational.

In [3]:
# Generate multiple completions to see the model's creativity
results = generator(
    "Satellite imagery reveals that",
    max_length=50,
    num_return_sequences=3,  # Generate 3 different continuations
    do_sample=True,          # Enable sampling (otherwise deterministic)
    temperature=0.8,
)

for i, result in enumerate(results, 1):
    print(f"--- Completion {i} ---")
    print(result['generated_text'])
    print()

Passing `generation_config` together with generation-related arguments=({'max_length', 'num_return_sequences', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- Completion 1 ---
Satellite imagery reveals that the US is the most common carrier of satellites to be tracked.

It is a good coincidence that the number of satellites tracked so far this year is the highest in four decades.

That may be because the US is still the world's most tracked space agency and its satellite record is still far from perfect.

The US has also had a long history of tracking satellites, with satellites that are now tracked by a variety of aircraft that include the Air Force's F-35 and the Navy's F-22.

'No one I know has ever been caught using a satellite,' said Tom Schiller, a former deputy secretary of state for communications, defense and civil aviation.

'But when you turn on your system, you actually have to get the right satellites to do this, and the US has an amazing history with satellites, so it's really only a matter of time before the next generation of satellites are connected to this platform.'

The report also found that at least 15 countries are

## 2. Under the Hood: Tokenizer + Model

The pipeline is convenient, but to really understand what's happening, let's break it apart into its two components:

1. **Tokenizer**: Converts text → token IDs (numbers the model can process)
2. **Model**: Takes token IDs → generates next token IDs

This is exactly what we learned about in the basics notebook, but now we can actually see the tensors.

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

print(f"Model: {model_name}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Vocabulary size: {tokenizer.vocab_size:,}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Model: gpt2
Parameters: 124,439,808
Vocabulary size: 50,257


### Step-by-Step Tokenization

Let's see exactly what the tokenizer does to your text. This demystifies the "magic" — the model never sees actual words, only numbers.

In [5]:
input_text = "Geospatial analysis using satellite data"

# Step 1: Tokenize
inputs = tokenizer(input_text, return_tensors="pt")

print(f"Original text: '{input_text}'")
print(f"\nToken IDs: {inputs['input_ids'][0].tolist()}")
print(f"Attention mask: {inputs['attention_mask'][0].tolist()}")
print(f"\nToken-by-token breakdown:")
for token_id in inputs['input_ids'][0]:
    token_str = tokenizer.decode([token_id])
    print(f"  {token_id.item():>6} → '{token_str}'")

Original text: 'Geospatial analysis using satellite data'

Token IDs: [10082, 2117, 34961, 3781, 1262, 11210, 1366]
Attention mask: [1, 1, 1, 1, 1, 1, 1]

Token-by-token breakdown:
   10082 → 'Ge'
    2117 → 'osp'
   34961 → 'atial'
    3781 → ' analysis'
    1262 → ' using'
   11210 → ' satellite'
    1366 → ' data'


**Notice how:**
- Some tokens have a leading space (like `' analysis'`)
- Long/uncommon words get split into sub-words
- The attention mask is all 1s (meaning "pay attention to all tokens")

### Step-by-Step Generation

In [6]:
# Step 2: Generate
with torch.no_grad():  # We're not training, no need to track gradients
    outputs = model.generate(
        **inputs, 
        max_new_tokens=30,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )

# Step 3: Decode back to text
full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(f"Input: '{input_text}'")
print(f"Full output: '{full_text}'")
print(f"\nGenerated {outputs.shape[1] - inputs['input_ids'].shape[1]} new tokens")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Input: 'Geospatial analysis using satellite data'
Full output: 'Geospatial analysis using satellite data.

We used the following algorithms:

Data extraction and analysis

To determine the maximum size of the dataset, we first use the'

Generated 30 new tokens


## 3. Generation Parameters Explained

HuggingFace gives you fine-grained control over how the model generates text. Here's what each parameter does:

In [7]:
prompt = "The future of remote sensing is"
inputs = tokenizer(prompt, return_tensors="pt")

# Greedy decoding — always picks the highest probability token
print("1. GREEDY (do_sample=False):")
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=30, do_sample=False)
print(f"   {tokenizer.decode(out[0], skip_special_tokens=True)}")
print()

# Sampling with low temperature — focused but slightly varied
print("2. LOW TEMPERATURE (0.3):")
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=30, do_sample=True, temperature=0.3)
print(f"   {tokenizer.decode(out[0], skip_special_tokens=True)}")
print()

# Sampling with high temperature — creative and diverse
print("3. HIGH TEMPERATURE (1.2):")
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=30, do_sample=True, temperature=1.2)
print(f"   {tokenizer.decode(out[0], skip_special_tokens=True)}")
print()

# With repetition penalty — prevents loops
print("4. WITH REPETITION PENALTY (1.3):")
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=30, do_sample=True, temperature=0.7, repetition_penalty=1.3)
print(f"   {tokenizer.decode(out[0], skip_special_tokens=True)}")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


1. GREEDY (do_sample=False):


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


   The future of remote sensing is in the hands of the next generation of sensors.

The next generation of sensors will be able to detect and track objects in the sky, and

2. LOW TEMPERATURE (0.3):


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


   The future of remote sensing is uncertain. The future of the world's oceans is uncertain. The future of the world's air is uncertain. The future of the world's forests is

3. HIGH TEMPERATURE (1.2):


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


   The future of remote sensing is not without its uncertainties. It isn't easy to envision a scenario where global temperatures can go as low as -5 degrees C, but our understanding of

4. WITH REPETITION PENALTY (1.3):
   The future of remote sensing is uncertain, and the field has also suffered from technical problems. A key concern for current methods was whether to increase or decrease sensitivity in a particular region due


### Parameters Reference

| Parameter | What it Does | Typical Range |
| :--- | :--- | :--- |
| `max_new_tokens` | Max tokens to generate | 10-2048 |
| `do_sample` | Enable random sampling vs greedy | True/False |
| `temperature` | Randomness level | 0.1 - 1.5 |
| `top_p` | Nucleus sampling threshold | 0.1 - 0.95 |
| `top_k` | Only sample from top K tokens | 10 - 100 |
| `repetition_penalty` | Penalize repeated tokens | 1.0 - 1.5 |
| `num_return_sequences` | How many completions to generate | 1-5 |
| `num_beams` | Beam search (explore multiple paths) | 1-5 |

## 4. Model Caching & Storage

When you first load a model, HuggingFace downloads it and caches it locally. This is important to know because models can be large.

- **Default cache**: `~/.cache/huggingface/hub/`
- **GPT-2 size**: ~500MB
- **Larger models**: Can be 5-50GB+

To manage your cache:

In [8]:
import os

cache_dir = os.path.expanduser("~/.cache/huggingface/hub/")
if os.path.exists(cache_dir):
    total_size = 0
    model_count = 0
    for item in os.listdir(cache_dir):
        item_path = os.path.join(cache_dir, item)
        if os.path.isdir(item_path) and item.startswith('models--'):
            model_name = item.replace('models--', '').replace('--', '/')
            size = sum(
                os.path.getsize(os.path.join(dp, f))
                for dp, _, files in os.walk(item_path)
                for f in files
            )
            total_size += size
            model_count += 1
            print(f"  📦 {model_name}: {size / 1024**2:.0f} MB")
    print(f"\nTotal: {model_count} models, {total_size / 1024**3:.1f} GB")
else:
    print("No HuggingFace cache found yet — it will be created when you first download a model.")

  📦 bert-base-uncased: 421 MB
  📦 cross-encoder/nli-distilroberta-base: 318 MB
  📦 distilbert/distilbert-base-uncased-finetuned-sst-2-english: 256 MB
  📦 facebook/bart-large-mnli: 0 MB
  📦 gpt2: 525 MB
  📦 sshleifer/distilbart-cnn-6-6: 879 MB

Total: 6 models, 2.3 GB


## 5. Comparison: HuggingFace vs Ollama

Now that you've used both, here's a honest comparison:

| Feature | HuggingFace | Ollama |
| :--- | :--- | :--- |
| **Ease of Use** | Moderate (scripting required) | Very easy (CLI/app) |
| **Flexibility** | Extremely high | Moderate |
| **Model Format** | Safetensors, PyTorch (.bin) | GGUF (quantized) |
| **Fine-tuning** | ✅ Full support | ❌ Not possible |
| **Quantization** | ✅ BitsAndBytes, GPTQ, AWQ | ✅ Built-in (GGUF) |
| **GPU Required?** | Depends on model size | No (CPU fallback) |
| **Best For** | Research, fine-tuning, custom architectures | Daily use, apps, quick prototyping |
| **Model Count** | 500,000+ on Hub | ~100 curated models |

**Bottom line:** Use Ollama for daily work and app building. Use HuggingFace when you need to fine-tune, do research, or access cutting-edge models.

## Challenge! 🚀

1. Search for **"TinyLlama"** on [huggingface.co](https://huggingface.co) and try replacing `gpt2` with its model path (`TinyLlama/TinyLlama-1.1B-Chat-v1.0`). How does the output quality compare?
2. Try the `pipeline('sentiment-analysis')` pipeline — this uses an encoder model (like BERT) instead of a decoder model. Do you notice the difference in how it works?

---

**Next notebook:** [RAG Basics](./rag_basics_ollama.ipynb) — Build a Retrieval-Augmented Generation system from scratch.